# Retrieval strategies benchmark on a Colab GPU

Runs `strategies_bench.py` (BM25 / dense / dense→rerank / hybrid RRF / hybrid RRF→rerank) on a Colab GPU, to compare against the M4 laptop.
Synthetic data only, so no private conversation content leaves the laptop.

1. **Runtime → Change runtime type →** pick a GPU (A100, L4, T4, …).
2. Run the cells in order. Cell 4 asks you to upload `strategies_bench.py` from `agenda-bench/`.
3. The 10k run takes a few minutes. The 1M run is mostly embedding time (estimate: well under an hour on an A100, several hours on a T4).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
# Postgres 17 from the official PGDG repo + pg_textsearch 1.5.1 built from source (BM25 index)
set -e
apt-get -qq update > /dev/null && apt-get -qq install -y curl ca-certificates lsb-release build-essential git > /dev/null
install -d /usr/share/postgresql-common/pgdg
curl -fsSo /usr/share/postgresql-common/pgdg/apt.postgresql.org.asc https://www.postgresql.org/media/keys/ACCC4CF8.asc
echo "deb [signed-by=/usr/share/postgresql-common/pgdg/apt.postgresql.org.asc] https://apt.postgresql.org/pub/repos/apt $(lsb_release -cs)-pgdg main" > /etc/apt/sources.list.d/pgdg.list
apt-get -qq update > /dev/null && apt-get -qq install -y postgresql-17 postgresql-server-dev-17 > /dev/null
rm -rf /tmp/pgts && git clone -q --depth 1 --branch v1.5.1 https://github.com/timescale/pg_textsearch /tmp/pgts
make -s -C /tmp/pgts PG_CONFIG=/usr/lib/postgresql/17/bin/pg_config > /dev/null && make -s -C /tmp/pgts PG_CONFIG=/usr/lib/postgresql/17/bin/pg_config install > /dev/null
cat > /etc/postgresql/17/main/conf.d/bench.conf <<EOF
port = 55434
shared_preload_libraries = 'pg_textsearch'
shared_buffers = '2GB'
maintenance_work_mem = '2GB'
work_mem = '256MB'
EOF
pg_ctlcluster 17 main restart
su postgres -c "psql -p 55434 -qc \"ALTER USER postgres PASSWORD 'bench'\""
su postgres -c "psql -p 55434 -tAc \"CREATE EXTENSION IF NOT EXISTS pg_textsearch; SELECT 'pg_textsearch ' || extversion FROM pg_extension WHERE extname = 'pg_textsearch'\""

In [ ]:
!pip -q install "psycopg[binary]" sentence-transformers

In [ ]:
from google.colab import files
files.upload()   # choose agenda-bench/strategies_bench.py

## 10k chunks (same size as the laptop report)

In [ ]:
!N_CHUNKS=10000 python strategies_bench.py > result_10k.json

In [ ]:
import json
def show(path):
    r = json.load(open(path))
    print(r["machine"], r["corpus"]["chunks"], "chunks |", r["index_build_s"])
    print(f"{'strategy':24} {'p50 ms':>9} {'p95 ms':>9}  recall@10  mrr@10  ndcg@10")
    for s, parts in r["latency"].items():
        q = r["quality"]["all"][s]
        print(f"{s:24} {parts['total']['p50_ms']:>9} {parts['total']['p95_ms']:>9}  {q['recall_at_10']:>9}  {q['mrr_at_10']:>6}  {q['ndcg_at_10']:>7}")
    for s, parts in r["latency"].items():
        if "rerank" in parts:
            print(f"  {s}: candidates {parts['candidates']}, rerank {parts['rerank']}")
show("result_10k.json")

## 1M chunks

Embeddings are saved every 50k chunks in `EMB_CACHE`. If the runtime disconnects, mount Google Drive and point `EMB_CACHE` there so a rerun resumes (needs ~4 GB of Drive space):

```python
from google.colab import drive; drive.mount('/content/drive')
```
then use `EMB_CACHE=/content/drive/MyDrive/emb_cache`.

In [ ]:
!N_CHUNKS=1000000 EMB_CACHE=/content/emb_cache python strategies_bench.py > result_1m.json

In [ ]:
show("result_1m.json")

In [ ]:
files.download("result_10k.json")
files.download("result_1m.json")